# RoBERTa

**Name:** Ankush Sarkar  
**Group:** Group 30  
**Codabench Link:** https://www.codabench.org/competitions/14898/

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Load Data

In [ ]:
TRAIN_SIZE = 50_000
DATA_DIR = "../bundle/public_data"

train_data = pd.read_csv(os.path.join(DATA_DIR, "training_data.csv")).iloc[:TRAIN_SIZE]
train_label = pd.read_csv(os.path.join(DATA_DIR, "training_label.csv")).iloc[:TRAIN_SIZE]
test_data = pd.read_csv(os.path.join(DATA_DIR, "testing_data.csv"))
test_label = pd.read_csv(os.path.join(DATA_DIR, "testing_label.csv"))

train_df = pd.concat([train_data, train_label], axis=1)
test_df = pd.concat([test_data, test_label], axis=1)

for df in [train_df, test_df]:
    df["question_title"] = df["question_title"].fillna("")
    df["question_content"] = df["question_content"].fillna("")
    df["class_index"] = df["class_index"] - 1  # shift 1-10 -> 0-9 (wht pytorch expects) 

print(f"Train: {len(train_df)} rows")
print(f"Test:  {len(test_df)} rows")
print(f"\nTrain label distribution:\n{train_df['class_index'].value_counts().sort_index()}")
print(f"\nTest label distribution:\n{test_df['class_index'].value_counts().sort_index()}")

## 2. Tokenization

In [ ]:
MODEL_NAME = "roberta-base"
MAX_LENGTH = 128
NUM_LABELS = 10

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

def tokenize_fn(batch):
    enc = tokenizer(
        batch["question_title"],
        batch["question_content"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    enc["labels"] = batch["class_index"]
    return enc

tokenized_train = train_dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["question_title", "question_content", "class_index"],
)
tokenized_test = test_dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["question_title", "question_content", "class_index"],
)

# 90/10 train-val split
split = tokenized_train.train_test_split(test_size=0.1, seed=SEED)
train_split = split["train"]
val_split = split["test"]

print(f"Train split: {len(train_split)}")
print(f"Val split:   {len(val_split)}")
print(f"Test set:    {len(tokenized_test)}")

## 3. Model Initialization and Training

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="roberta_results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    logging_steps=100,
    save_total_limit=2,
    seed=SEED,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_split,
    eval_dataset=val_split,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

## 4. Evaluation on Test Set

In [ ]:
pred_output = trainer.predict(tokenized_test)
preds = np.argmax(pred_output.predictions, axis=-1)
true_labels = np.array(tokenized_test["labels"])

test_accuracy = accuracy_score(true_labels, preds)
test_f1_macro = f1_score(true_labels, preds, average="macro")
test_f1_weighted = f1_score(true_labels, preds, average="weighted")

print(f"Test Accuracy:    {test_accuracy:.4f}")
print(f"Test F1 (macro):  {test_f1_macro:.4f}")
print(f"Test F1 (weighted): {test_f1_weighted:.4f}")

In [ ]:
LABEL_NAMES = [
    "Society & Culture",
    "Science & Mathematics",
    "Health",
    "Education & Reference",
    "Computers & Internet",
    "Sports",
    "Business & Finance",
    "Entertainment & Music",
    "Family & Relationships",
    "Politics & Government",
]

print(classification_report(true_labels, preds, target_names=LABEL_NAMES, digits=4))

## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(true_labels, preds, labels=list(range(NUM_LABELS)))

SHORT_LABELS = ["S&C", "Sci", "Hlt", "Edu", "C&I", "Spt", "B&F", "Ent", "Fam", "Pol"]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=SHORT_LABELS, yticklabels=SHORT_LABELS, ax=ax)
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.set_title("RoBERTa Confusion Matrix on Test Set")
plt.tight_layout()
plt.savefig("confusion_matrix_roberta.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nConfusion matrix (rows = true, columns = predicted):")
cm_df = pd.DataFrame(cm, index=SHORT_LABELS, columns=SHORT_LABELS)
print(cm_df.to_string())

## 6. Generate Submission File

In [ ]:
# Shift predictions back to 1-indexed for submission
preds_1indexed = preds + 1

submission_df = pd.DataFrame({"class_index": preds_1indexed})
submission_df.to_csv("testing_label.csv", index=False)

print(f"Saved testing_label.csv with {len(submission_df)} predictions")
print(f"Prediction distribution:\n{submission_df['class_index'].value_counts().sort_index()}")

# Verify against ground truth (also 1-indexed)
gt = pd.read_csv(os.path.join(DATA_DIR, "testing_label.csv"))
verify_acc = accuracy_score(gt["class_index"].values, preds_1indexed)
print(f"\nVerification accuracy against ground truth: {verify_acc:.4f}")

## 7. Per-Class Analysis (for Error Analysis)

In [ ]:
# Per-class F1, precision, recall
per_class_f1 = f1_score(true_labels, preds, average=None)
from sklearn.metrics import precision_score, recall_score

per_class_precision = precision_score(true_labels, preds, average=None)
per_class_recall = recall_score(true_labels, preds, average=None)

analysis_df = pd.DataFrame({
    "Category": LABEL_NAMES,
    "Precision": per_class_precision,
    "Recall": per_class_recall,
    "F1": per_class_f1,
}).round(4)

print(analysis_df.to_string(index=False))

# Strongest and weakest categories
best_idx = np.argmax(per_class_f1)
worst_idx = np.argmin(per_class_f1)
print(f"\nBest:  {LABEL_NAMES[best_idx]} (F1: {per_class_f1[best_idx]:.4f})")
print(f"Worst: {LABEL_NAMES[worst_idx]} (F1: {per_class_f1[worst_idx]:.4f})")

# Top misclassification pairs
print("\nTop misclassification pairs (true -> predicted, count):")
misclass_pairs = []
for i in range(NUM_LABELS):
    for j in range(NUM_LABELS):
        if i != j and cm[i][j] > 0:
            misclass_pairs.append((LABEL_NAMES[i], LABEL_NAMES[j], cm[i][j]))
misclass_pairs.sort(key=lambda x: x[2], reverse=True)
for true_cat, pred_cat, count in misclass_pairs[:10]:
    print(f"  {true_cat:30s} -> {pred_cat:30s}: {count}")